In [ ]:
# ============================================================
# GuardAI — Community App: User Input Spam Detector
# User types a message → AI routes it to Inbox or Spam
# ============================================================

!pip install pandas scikit-learn -q

import pandas as pd
import json
import os
from datetime import datetime
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier

# ============================================================
# STEP 1 — Built-in Training Data (No CSV needed)
# ============================================================

TRAINING_DATA = [
    # ── SPAM ──
    ("URGENT HIRING: Earn $1000/week from home. No experience needed. Apply NOW!", "SPAM"),
    ("FREE 2BR apartment $400/month. No credit check! Contact immediately!", "SPAM"),
    ("GOVERNMENT GRANT: You qualify for $5000 community assistance. Claim today!", "SPAM"),
    ("Join our wellness team! Sell products, recruit friends, earn unlimited income!", "SPAM"),
    ("WARNING: Tap water contains dangerous chemicals. Buy our filter NOW!", "SPAM"),
    ("Your community account has been suspended. Verify your identity immediately.", "SPAM"),
    ("Need cash fast? Get $5000 loan in 10 minutes. No credit check. 100% approved!", "SPAM"),
    ("Donate to help flood victims. 100% goes to families. Send via PayPal now!", "SPAM"),
    ("FREE community BBQ! Just register and pay a $20 booking fee to confirm!", "SPAM"),
    ("Work from home stuffing envelopes and earn $400 weekly. Unlimited positions!", "SPAM"),
    ("BREAKING: 5G towers causing illness in your neighborhood. Protect family now!", "SPAM"),
    ("You are eligible for free groceries from our community program. Register now!", "SPAM"),
    ("Payday loan — borrow $1000 today. Instant approval. Anyone qualifies!", "SPAM"),
    ("Hot singles in your area want to meet you tonight! Click here!", "SPAM"),
    ("Data entry job $50/hour. No skills required. Start today! Apply here!", "SPAM"),
    ("EXCLUSIVE: Community leadership seminar FREE tickets but pay $50 deposit!", "SPAM"),
    ("Local orphanage urgently needs funds. Send directly to our account now!", "SPAM"),
    ("Network marketing opportunity — share with 5 friends and earn $500/week!", "SPAM"),
    ("Council approved $1500 utility bill relief for your area. Apply immediately!", "SPAM"),
    ("Stop posting here or you will regret it. Consider this your final warning.", "SPAM"),
    ("URGENT: $3000 unclaimed government benefit in your name. Verify to receive!", "SPAM"),
    ("Microchips being injected via flu shots. Refuse vaccination immediately!", "SPAM"),
    ("FREE self-defence class for women. Register now — just pay $30 deposit!", "SPAM"),
    ("Your community benefit payment failed. Update bank details to receive it.", "SPAM"),
    ("I earn $5000/month helping community members. I can show you how. Join me!", "SPAM"),
    ("ACT NOW: Community ambassador role — $50 per post. Apply before midnight!", "SPAM"),
    ("Emergency rental assistance $2000 available. Apply immediately no docs needed!", "SPAM"),
    ("ALERT: Local hospital giving unnecessary treatments for profit. Avoid them!", "SPAM"),
    ("Community investment club meeting — guaranteed returns discussed inside!", "SPAM"),
    ("Get $10,000 loan in 24 hours. We say YES when banks say no. Apply today!", "SPAM"),
    ("FREE iPhone 15 giveaway just pay $1 shipping! Limited to first 100 people!", "SPAM"),
    ("You have been pre-approved for $50,000 loan no credit check apply now!", "SPAM"),
    ("LAST CHANCE: Double your investment in crypto. 200% returns guaranteed!", "SPAM"),
    ("Your kind is not welcome here. Get out of our community now immediately.", "SPAM"),
    ("Cheap rentals $300/month all inclusive contact us immediately limited slots!", "SPAM"),
    ("Be your own boss join our community sales team earn unlimited commissions!", "SPAM"),
    ("Your community app account flagged verify identity here urgently or lose access!", "SPAM"),
    ("Natural herb cures illness in 3 days doctors wont tell you buy it here!", "SPAM"),
    ("HIRING NOW earn $500 day from home no experience needed apply immediately!", "SPAM"),
    ("Fundraiser sick child community send funds via Western Union today urgent!", "SPAM"),

    # ── HAM ──
    ("Community park cleanup this Saturday 9am. Gloves provided. All welcome!", "HAM"),
    ("Does anyone know a reliable plumber in the north end of town?", "HAM"),
    ("Free vegetables from my garden. Free to anyone who needs them.", "HAM"),
    ("Reminder: neighborhood watch meeting this Thursday at 7pm.", "HAM"),
    ("Support group for grief and loss meets every Monday evening at 6:30pm.", "HAM"),
    ("Lost cat — ginger tabby named Socks. Last seen near the park.", "HAM"),
    ("Can anyone recommend a good after-school program for a 9-year-old?", "HAM"),
    ("We need volunteers to deliver meals to isolated seniors next Wednesday.", "HAM"),
    ("Free tutoring offered to high school students struggling with math.", "HAM"),
    ("Road closed on Bridge Street due to flooding. Use Oak Ave instead.", "HAM"),
    ("If you or someone you know is in crisis please call Lifeline on 13 11 14.", "HAM"),
    ("The community center will be closed next Monday for the public holiday.", "HAM"),
    ("Huge thank you to everyone who helped at the community cleanup yesterday!", "HAM"),
    ("Free family movie night at the park on Friday. Bring your own chairs!", "HAM"),
    ("Monthly neighborhood watch meeting this Thursday at 7pm at the hall.", "HAM"),
    ("Spare bicycle in working order — free to a community member who needs it.", "HAM"),
    ("Storm warning tonight. Secure outdoor furniture and check on your neighbors.", "HAM"),
    ("Donating a working laptop to a student who needs one for school. Message me.", "HAM"),
    ("Free mindfulness sessions at the library every Wednesday at noon.", "HAM"),
    ("Our food drive collected over 800 items this month. Thank you all!", "HAM"),
    ("Phone scam targeting elderly residents in our area. Please warn your family.", "HAM"),
    ("Council bin collection is running one day late this week due to the holiday.", "HAM"),
    ("Multicultural food fair this Sunday celebrating our diverse neighborhood!", "HAM"),
    ("Looking for a mentor for a 16-year-old interested in coding and technology.", "HAM"),
    ("Free first aid course offered to community members. Register by Thursday.", "HAM"),
    ("Water outage expected from 9am to 2pm tomorrow on the south side of town.", "HAM"),
    ("I have surplus home-cooked food. Happy to share with a family who needs it.", "HAM"),
    ("Book swap at the library on Saturday. Bring one take one — completely free!", "HAM"),
    ("Postnatal support group every Friday at 10am. All new parents welcome!", "HAM"),
    ("Please slow down near Park Lane — children are playing there every afternoon.", "HAM"),
    ("Local school wins environmental award. Amazing effort from students and staff!", "HAM"),
    ("Free English conversation practice offered by retired teacher with spare time.", "HAM"),
    ("Stray dog spotted near the school. Animal control has been notified already.", "HAM"),
    ("Seeking volunteers to read to children at the library every Thursday at 4pm.", "HAM"),
    ("New bus route 47 now stops directly outside the community health center.", "HAM"),
    ("Can anyone help a recently arrived refugee family navigate local services?", "HAM"),
    ("Annual street party is back this Sunday. Bring a plate to share!", "HAM"),
    ("Community resilience program free six-week course starts next month register.", "HAM"),
    ("Men's wellbeing morning tea come chat connect and support each other free.", "HAM"),
    ("Missing elderly man last seen near shopping center please call 000 if seen.", "HAM"),
]

# Train model
df = pd.DataFrame(TRAINING_DATA, columns=["text", "label"])

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 3), max_features=10000,
        sublinear_tf=True, min_df=1, analyzer="char_wb"
    )),
    ("clf", SGDClassifier(
        loss="modified_huber", max_iter=1000,
        random_state=42, class_weight="balanced"
    ))
])
model.fit(df["text"], df["label"])
print("✅ Model ready!\n")

# ============================================================
# STEP 2 — Inbox & Spam Folder
# ============================================================

inbox       = []   # clean messages
spam_folder = []   # blocked messages

def save_folders():
    with open("inbox.json", "w") as f:
        json.dump(inbox, f, indent=2)
    with open("spam_folder.json", "w") as f:
        json.dump(spam_folder, f, indent=2)

# ============================================================
# STEP 3 — Route a Message
# ============================================================

def route(sender, message):
    probs     = model.predict_proba([message])[0]
    classes   = model.classes_
    scores    = dict(zip(classes, probs))
    label     = max(scores, key=scores.get)
    confidence = scores[label] * 100
    spam_pct  = round(scores.get("SPAM", 0) * 100, 1)
    ham_pct   = round(scores.get("HAM",  0) * 100, 1)

    entry = {
        "id"        : len(inbox) + len(spam_folder) + 1,
        "sender"    : sender,
        "message"   : message,
        "label"     : label,
        "spam_pct"  : spam_pct,
        "ham_pct"   : ham_pct,
        "time"      : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    if label == "SPAM":
        spam_folder.append(entry)
        print_spam_block(entry)
    else:
        inbox.append(entry)
        print_inbox_delivery(entry)

    save_folders()
    return label

# ============================================================
# STEP 4 — Display
# ============================================================

def bar(score, width=20):
    filled = int(score / 100 * width)
    return "█" * filled + "░" * (width - filled)

def print_spam_block(e):
    print("\n" + "━" * 58)
    print("  🚫  SPAM DETECTED — MESSAGE BLOCKED")
    print("━" * 58)
    print(f"  From    : {e['sender']}")
    print(f"  Time    : {e['time']}")
    print(f"  Message : {e['message'][:65]}{'...' if len(e['message'])>65 else ''}")
    print(f"\n  🔴 Spam  {bar(e['spam_pct'])} {e['spam_pct']}%")
    print(f"  🟢 Clean {bar(e['ham_pct'])}  {e['ham_pct']}%")
    print(f"\n  ⛔ This message was sent to your SPAM FOLDER.")
    print(f"     The user will NOT see this message.")
    print("━" * 58 + "\n")

def print_inbox_delivery(e):
    print("\n" + "━" * 58)
    print("  ✅  CLEAN MESSAGE — DELIVERED TO INBOX")
    print("━" * 58)
    print(f"  From    : {e['sender']}")
    print(f"  Time    : {e['time']}")
    print(f"  Message : {e['message'][:65]}{'...' if len(e['message'])>65 else ''}")
    print(f"\n  🟢 Clean {bar(e['ham_pct'])} {e['ham_pct']}%")
    print(f"  🔴 Spam  {bar(e['spam_pct'])}  {e['spam_pct']}%")
    print(f"\n  📥 Message successfully delivered to inbox.")
    print("━" * 58 + "\n")

def view_inbox():
    print("\n" + "═" * 58)
    print(f"  📥  INBOX  —  {len(inbox)} message(s)")
    print("═" * 58)
    if not inbox:
        print("  (empty)\n"); return
    for m in inbox:
        print(f"\n  #{m['id']} | From: {m['sender']} | {m['time']}")
        print(f"  ✅ Clean {m['ham_pct']}%")
        print(f"  📩 {m['message']}")
        print("  " + "─" * 50)
    print()

def view_spam():
    print("\n" + "═" * 58)
    print(f"  🗑️   SPAM FOLDER  —  {len(spam_folder)} message(s) blocked")
    print("═" * 58)
    if not spam_folder:
        print("  (empty)\n"); return
    for m in spam_folder:
        print(f"\n  #{m['id']} | From: {m['sender']} | {m['time']}")
        print(f"  🚫 Spam {m['spam_pct']}%")
        print(f"  📩 {m['message'][:80]}{'...' if len(m['message'])>80 else ''}")
        print("  " + "─" * 50)
    print()

def summary():
    total = len(inbox) + len(spam_folder)
    pct   = round(len(spam_folder)/total*100, 1) if total else 0
    print(f"\n  📊 Total received : {total}")
    print(f"  📥 Inbox          : {len(inbox)} clean messages")
    print(f"  🗑️  Spam blocked   : {len(spam_folder)} messages ({pct}%)\n")

def not_spam(idx):
    """Move item from spam back to inbox + retrain model."""
    match = next((m for m in spam_folder if m["id"] == idx), None)
    if not match:
        print("⚠️  Message not found."); return
    spam_folder.remove(match)
    match["label"] = "HAM"
    inbox.append(match)
    save_folders()
    global df, model
    new = pd.DataFrame([{"text": match["message"], "label": "HAM"}])
    df  = pd.concat([df, new], ignore_index=True)
    model.fit(df["text"], df["label"])
    print(f"✅ Moved #{idx} to inbox. Model updated — won't flag this again.")

def mark_spam(idx):
    """Move item from inbox to spam + retrain model."""
    match = next((m for m in inbox if m["id"] == idx), None)
    if not match:
        print("⚠️  Message not found."); return
    inbox.remove(match)
    match["label"] = "SPAM"
    spam_folder.append(match)
    save_folders()
    global df, model
    new = pd.DataFrame([{"text": match["message"], "label": "SPAM"}])
    df  = pd.concat([df, new], ignore_index=True)
    model.fit(df["text"], df["label"])
    print(f"✅ Moved #{idx} to spam. Model updated — will catch this next time.")

# ============================================================
# STEP 5 — Live Input Loop
# ============================================================

print("=" * 58)
print("  📬  COMMUNITY APP — SMART MESSAGE ROUTER")
print("  Type a message → AI decides: Inbox or Spam")
print("=" * 58)
print("""
  Commands:
  Type any message      → AI checks and routes it
  !inbox                → view inbox
  !spam                 → view spam folder
  !summary              → message counts
  !notspam <id>         → move spam back to inbox
  !markspam <id>        → move inbox item to spam
  !quit                 → exit
""")

while True:
    try:
        msg = input("✏️  Enter message: ").strip()

        if not msg:
            continue

        # ── Commands ──────────────────────
        if msg.lower() in ("!quit", "quit", "exit"):
            summary()
            print("👋 Goodbye! All messages saved.\n")
            break

        elif msg.lower() == "!inbox":
            view_inbox()

        elif msg.lower() == "!spam":
            view_spam()

        elif msg.lower() == "!summary":
            summary()

        elif msg.lower().startswith("!notspam "):
            try:   not_spam(int(msg.split()[1]))
            except: print("⚠️  Usage: !notspam <id number>")

        elif msg.lower().startswith("!markspam "):
            try:   mark_spam(int(msg.split()[1]))
            except: print("⚠️  Usage: !markspam <id number>")

        # ── Route the message ─────────────
        else:
            sender = input("  👤 Sender name : ").strip() or "Anonymous"
            route(sender, msg)

    except KeyboardInterrupt:
        print("\n\n👋 Saved. Goodbye!")
        break
    except Exception as e:
        print(f"❌ Error: {e}")


✅ Model ready!

  📬  COMMUNITY APP — SMART MESSAGE ROUTER
  Type a message → AI decides: Inbox or Spam

  Commands:
  Type any message      → AI checks and routes it
  !inbox                → view inbox
  !spam                 → view spam folder
  !summary              → message counts
  !notspam <id>         → move spam back to inbox
  !markspam <id>        → move inbox item to spam
  !quit                 → exit

✏️  Enter message: 1000$ free bonus , get free gifts and chocolates
  👤 Sender name : !spam

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅  CLEAN MESSAGE — DELIVERED TO INBOX
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  From    : !spam
  Time    : 2026-03-28 22:05:20
  Message : 1000$ free bonus , get free gifts and chocolates

  🟢 Clean ███████████████████░ 95.6%
  🔴 Spam  ░░░░░░░░░░░░░░░░░░░░  4.4%

  📥 Message successfully delivered to inbox.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

✏️  Enter message: $100 free
  👤 Sende